# Training MNB model

In [53]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [54]:
df = pd.read_csv("data/merged_data.csv", index_col="date", parse_dates=True)
df.sort_index(inplace=True)

We define our features (X) and target (y)

Our features (X) is comprised of our cleaned text data (X1) and our financial data (X2).

In [55]:
X = df[["summary", "pct_change", "volume"]]
y = df["target"]

Now we split our data for training and testing

DO NOT SHUFFLE because we don't want to mix time series data

In [56]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

In [57]:
print(f"Training Range: {X_train.index.min()} to {X_train.index.max()}")
print(f"Testing Range:  {X_test.index.min()} to {X_test.index.max()}")

Training Range: 2017-12-19 00:00:00 to 2020-01-10 00:00:00
Testing Range:  2020-01-13 00:00:00 to 2020-07-17 00:00:00


Now we have two different types of data that we need to handle differently

1. Text data - we will use TF-IDF vectorization to convert text into numerical format
2. Numerical data - we will use MinMaxScaler to normalize the data

The reason we need two different scalers is because Naive Bayes won't work with negative values

In [58]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.naive_bayes import MultinomialNB

In [59]:
text_features = "summary"
numerical_features = ["pct_change", "volume"]

Make our preprocessor for the vectorization and normalization steps

In [60]:
preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2)), text_features),
        ("scaler", MinMaxScaler(clip=True), numerical_features)
    ]
)

Setup our pipeline with the preprocessor and the Naive Bayes classifier

In [61]:
pipeline_mnb = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', MultinomialNB(alpha=0.1, fit_prior=False))
])

In [62]:
pipeline_mnb.fit(X_train, y_train)

,steps,"[('preprocessor', ...), ('classifier', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('tfidf', ...), ('scaler', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [63]:
y_pred = pipeline_mnb.predict(X_test)

from sklearn.metrics import accuracy_score, classification_report
print("Accuracy:", accuracy_score(y_test, y_pred))

print(classification_report(y_test, y_pred))

Accuracy: 0.5461538461538461
              precision    recall  f1-score   support

         0.0       0.48      0.26      0.34        58
         1.0       0.57      0.78      0.65        72

    accuracy                           0.55       130
   macro avg       0.52      0.52      0.50       130
weighted avg       0.53      0.55      0.51       130



The model predicts up most of the tie and guesses it right 54% of the time it is not good at guessing the down trend in the market

In [64]:
%%capture
%pip install joblib

In [ ]:
# exporting the baseline model to compare after we tune it
import joblib
joblib.dump(pipeline_mnb, "models/mnb_baseline.pkl")

['models/mnb_pipeline.pkl']